In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import numpy as np
from torch.utils.data import DataLoader, Subset
import torch
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score,
    precision_score, f1_score
)

from dataset.dataset import Dataset
from utils.config import load_config
from utils.builder import build_trainer
from utils.metrics import accuracy, sensitivity, specificity, roc_auc

In [2]:
config_path = "../configs/fcn_full.yaml"

In [3]:
config = load_config(config_path)
data_path = "../"+config["data"]["data_path"]

torch.manual_seed(config["experiment"]["seed"])
np.random.seed(config["experiment"]["seed"])

dataset = Dataset(data_path, "G", config["data"]["batch_size"], use_tabular=config["data"]["use_tabular"], tabular_features=config["data"]["tabular_features"])
train_loader, val_loader, test_loader = dataset.get_loaders()

dataset_f = Dataset(data_path, "F", config["data"]["batch_size"])
_, val_loader_f, test_loader_f = dataset_f.get_loaders()

dataset_m = Dataset(data_path, "M", config["data"]["batch_size"])       
_, val_loader_m, test_loader_m = dataset_m.get_loaders()

In [4]:
trainer = build_trainer(config)
trainer.load_checkpoint()

In [5]:
def bootstrap_sample(loader):
    dataset = loader.dataset
    n = len(dataset)
    indices = np.random.choice(n, n, replace=True)
    return DataLoader(Subset(dataset, indices), batch_size=loader.batch_size)

In [6]:
def compute_bootstrap_metrics(loader, model, metric, num_samples=100, threshold=None):
    metric_values = []
    
    for _ in tqdm(range(num_samples)):
        boot_loader = bootstrap_sample(loader)
        if threshold is not None:
            ans = metric(boot_loader, model, threshold)
        else:
            ans = metric(boot_loader, model)
        metric_values.append(ans)
        
    return np.array(metric_values)

In [7]:
def check_bias(model, metric, loader1, loader2, num_samples=10, ci=0.95, threshold=None):
    metrics1 = compute_bootstrap_metrics(loader1, model, metric, num_samples=num_samples, threshold=threshold)
    metrics2 = compute_bootstrap_metrics(loader2, model, metric, num_samples=num_samples, threshold=threshold)
    diff = metrics1 - metrics2
    
    lower = np.percentile(diff, (1 - ci) / 2 * 100)
    upper = np.percentile(diff, (1 + ci) / 2 * 100)
    
    print("mean_diff:", diff.mean(), "ci_lower:", lower, "ci_upper:", upper)

    if 0 < lower or 0 > upper:
        print("Bias detected")
    else:
        print("No bias detected")

In [8]:
best_thr = trainer.choose_threshold(val_loader)

/home/maryna/anaconda3/envs/compmed/lib/python3.9/site-packages/torch/nn/modules/conv.py:366: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1027.)
  return F.conv1d(


Best threshold: 0.404
Sensitivity: 0.902
Specificity: 0.816
AUC: 0.954
Accuracy: 0.858


### ROC-AUC


In [9]:
check_bias(trainer.model, roc_auc, test_loader_f, test_loader_m, num_samples=100, ci=0.95)

100%|██████████| 100/100 [03:36<00:00,  2.17s/it]

mean_diff: -0.0363736232894945 ci_lower: -0.14341021949308796 ci_upper: 0.05343672797198056
No bias detected


### Accuracy

In [10]:
check_bias(trainer.model, accuracy, test_loader_f, test_loader_m, num_samples=100, ci=0.95, threshold=best_thr)

100%|██████████| 100/100 [03:34<00:00,  2.15s/it]

mean_diff: -0.06960959409594095 ci_lower: -0.10627117378316639 ci_upper: -0.04083878053066241
Bias detected


### Sensitivity

In [11]:
check_bias(trainer.model, sensitivity, test_loader_f, test_loader_m, num_samples=100, ci=0.95, threshold=best_thr)

100%|██████████| 100/100 [03:28<00:00,  2.09s/it]

mean_diff: -0.039628119719093705 ci_lower: -0.22709129106187928 ci_upper: 0.1801522435897433
No bias detected


### Specificity

In [12]:
check_bias(trainer.model, specificity, test_loader_f, test_loader_m, num_samples=100, ci=0.95, threshold=best_thr)

100%|██████████| 100/100 [03:36<00:00,  2.16s/it]

mean_diff: -0.08514677488011113 ci_lower: -0.25512031072375896 ci_upper: 0.08492348873847587
No bias detected
